In [ ]:
%load_ext autoreload
%autoreload 2
from module_scripts import datahandler
from automated_scripts import nemo_3_project_img, nemo_4_extract_nematic, nemo_morph_curvature
from gastruloids import nemo_cylindrical_coords, nemo_cylindrical_analyse_nematic
import glob
from scipy.stats import mannwhitneyu
from scipy.interpolate import UnivariateSpline
from skimage.transform import rescale
from module_scripts import analysis
from collections import defaultdict
import matplotlib.colors as mcolors
from scipy.interpolate import CubicSpline
import pandas as pd
from matplotlib.ticker import MaxNLocator
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import numpy as np
import os
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.cm as cm
import re
import matplotlib.gridspec as gridspec

plt.style.use('default')

## Visual Phase Diagrams

In [ ]:
# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
REFERENCE_SCALE = 0.3  # Target microns per pixel (GLOBAL SPATIAL RESOLUTION)
SAVE_DIR = '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Illustrator/gastruloids_k-andreadis/Links/_EXP4_cleaned/overview'
DATA_DIR = '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/EXP4_CLEANED_EXAMPLES/'

paths_200 = [
    '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/EXP4_CLEANED_EXAMPLES/72h_200_Gas1.tif',
    '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/EXP4_CLEANED_EXAMPLES/96h_200_Gas1.tif',
    '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/EXP4_CLEANED_EXAMPLES/104h_200_Gas6.tif',
    '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/EXP4_CLEANED_EXAMPLES/112h_200_Gas2.tif',
    '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/EXP4_CLEANED_EXAMPLES/120h_200_Gas1.tif'
]

paths_300 = [
    '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/EXP4_CLEANED_EXAMPLES/72h_300_Gas2.tif',
    '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/EXP4_CLEANED_EXAMPLES/96h_300_Gas8.tif',
    '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/EXP4_CLEANED_EXAMPLES/104h_300_Gas1.tif',
    '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/EXP4_CLEANED_EXAMPLES/112h_300_Gas2.tif',
    '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/EXP4_CLEANED_EXAMPLES/120h_300_Gas2.tif'
]

grid_paths = [paths_200, paths_300]
time_labels = ["72hps", "96hps", "104hps", "112hps", "120hps"]
size_labels = [r"\approx 200\text{ cells}", r"\approx 300\text{ cells}"]

# ==========================================
# 2. PHYSICAL NORMALIZATION (SPATIAL & INTENSITY)
# ==========================================
processed_grid = []
global_max_val = 0

print(">> STARTING PHYSICAL NORMALIZATION...")

for row_paths in grid_paths:
    row_data = []
    for p in row_paths:
        img_load = analysis.load_img_virtual(path=p, t_sel_idx=0, c_sel_idx=0)
        img_raw, _, img_scale, _ = img_load
        current_res = img_scale[1]
        sf = current_res / REFERENCE_SCALE
        max_proj = np.max(img_raw, axis=0)
        img_rescaled = rescale(max_proj, sf, anti_aliasing=True, preserve_range=True)
        local_99 = np.percentile(img_rescaled, 99.5)
        if local_99 > global_max_val:
            global_max_val = local_99
        row_data.append(img_rescaled)
        print(f"Processed {os.path.basename(p)}: Original Res {current_res:.3f} -> Factor {sf:.2f}")
    processed_grid.append(row_data)

print(">> Normalization Complete. All images now share the exact same micron-to-pixel ratio.")

# ==========================================
# 3. POLISHED VISUALIZATION (CENTERED & UNIFIED)
# ==========================================

cmap = "Greys"

# 1. DEFINE THE UNIFIED CANVAS (in pixels)
# We add a small buffer (100px) so the largest image isn't touching the edges
all_h = [img.shape[0] for row in processed_grid for img in row]
all_w = [img.shape[1] for row in processed_grid for img in row]
canvas_h, canvas_w = max(all_h) + 100, max(all_w) + 100

plt.rcParams.update({'font.family': 'sans-serif', 'svg.fonttype': 'none'})
fig, axes = plt.subplots(2, 5, figsize=(18, 7), constrained_layout=True, facecolor='white')
im = None
for r, row_data in enumerate(processed_grid):
    for c, img in enumerate(row_data):
        ax = axes[r, c]

        h, w = img.shape
        # Calculate offsets to center the image on the unified canvas
        off_y = (canvas_h - h) // 2
        off_x = (canvas_w - w) // 2

        # Display image with an 'extent' to center it in the coordinate system
        # extent = [left, right, bottom, top]
        im = ax.imshow(img, cmap=cmap, vmin=0, vmax=global_max_val,
                       origin='upper', extent=[off_x, off_x + w, off_y + h, off_y])

        # Set unified limits so all "boxes" are the same physical size
        ax.set_xlim(0, canvas_w)
        ax.set_ylim(canvas_h, 0)

        # Style tweaks
        if r == 0:
            ax.set_title(fr"$\mathbf{{{time_labels[c]}}}$", color='black', fontsize=24, pad=15)
        if c == 0:
            ax.set_ylabel(fr"$\mathbf{{{size_labels[r]}}}$", color='black', fontsize=20, labelpad=10)

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_facecolor('white')
        for spine in ax.spines.values():
            spine.set_visible(False)

# ==========================================
# 2. THE SCALE BAR (Bold & Shifted Up)
# ==========================================

# This forces the math parser to support bold Greek symbols like \mu
plt.rcParams.update({
    'mathtext.fontset': 'stix',
    'mathtext.default': 'rm'
})

BAR_UM = 300
bar_px = BAR_UM / REFERENCE_SCALE
ax_sb = axes[-1, -1]

# Padding configuration (Increase these to move the bar away from the edges)
pad_from_right = 150
pad_from_bottom = 150  # Moves the whole unit UP
text_above_bar = 40  # Gap between text and bar

# Calculate coordinates
bar_x_end = canvas_w - pad_from_right
bar_x_start = bar_x_end - bar_px
bar_y = canvas_h - pad_from_bottom

# 1. The Bar
ax_sb.plot([bar_x_start, bar_x_end], [bar_y, bar_y],
           color='black', lw=7, solid_capstyle='butt', zorder=10)

# 2. The Label (Moved backslash out of the {expression} to avoid SyntaxError)
# We use \\mu to ensure the backslash reaches the LaTeX renderer
ax_sb.text(bar_x_start + (bar_px / 2),
           bar_y - text_above_bar,
           fr"$\mathbf{{{BAR_UM}}}\ \mathbf{{um}}$",
           color='black', ha='center', va='bottom',
           fontsize=26, zorder=10)
# 3. THE COLORBAR
# Shared across the whole figure
cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.5, aspect=30, pad=0.03)
cbar.outline.set_edgecolor('black')
cbar.set_label(r"$\mathbf{Fluorescence\ Intensity\ (a.u.)}$", fontsize=18, labelpad=15)
cbar.ax.tick_params(labelsize=14)

# 4. FINAL EXPORT
os.makedirs(SAVE_DIR, exist_ok=True)
final_path = os.path.join(SAVE_DIR, 'max-projections_phase-diagram.pdf')
plt.savefig(final_path, dpi=500, facecolor='white', bbox_inches='tight')

print(f">> Publication-quality figure saved: {final_path}")
plt.show()

## Batch Setup

In [ ]:
img_path_list_exp4_cleaned = sorted(glob.glob(
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/**/*.tif',
    recursive=True))
print(f"Found {len(img_path_list_exp4_cleaned)} images !")
img_path_list_exp4_cleaned_only72h = [p for p in img_path_list_exp4_cleaned if p.split(os.sep)[-3] == '72']
img_path_list_exp4_cleaned_only120h = [p for p in img_path_list_exp4_cleaned if p.split(os.sep)[-3] == '120']
t_select = 0
c_select = 0

In [ ]:
gastruloid_batch_output_folder = '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/URI_25022026_PR_NEMO/EXP4_filter_membrane/!batch-analysis'
dataset_morphology_name = "df_morpho.pkl"
# q_decomp_radius_selected = 50.0
q_decomp_radius_selected = 100.0
# layer_label_selected = "sampling_mesh_proj_4.95_to_5.05_um_mean"
# layer_label_selected = 'sampling_mesh_proj_14.95_to_15.05_um_mean'
layer_label_selected = "sampling_mesh_proj_24.95_to_25.05_um_mean"
q_decomp_label_selected = f"r-{q_decomp_radius_selected}um"
dataset_nematic_binned_name = f"df_nematic_binned_{layer_label_selected}_{q_decomp_label_selected}.pkl"
dataset_nematic_profile_name = f"df_nematic_{layer_label_selected}_{q_decomp_label_selected}.pkl"
morphology_batch_analysis_path = os.path.join(gastruloid_batch_output_folder, "morphology")
os.makedirs(morphology_batch_analysis_path, exist_ok=True)
nematic_batch_analysis_path = os.path.join(gastruloid_batch_output_folder, "nematic")
os.makedirs(nematic_batch_analysis_path, exist_ok=True)
morpho_nematic_batch_analysis_path = os.path.join(gastruloid_batch_output_folder, "combined-morpho-nematic")
os.makedirs(morpho_nematic_batch_analysis_path, exist_ok=True)

## Quick Batch Visual

In [ ]:
# ==========================================
# 1. AUTO-GROUPING & CONFIG
# ==========================================
REFERENCE_SCALE = 0.3
BAR_UM = 300
SAVE_DIR = '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Illustrator/gastruloids_k-andreadis/Links/_EXP4_cleaned/overview'
time_points = ['72', '96', '104', '112', '120']
sizes = ['200', '300']
data_groups = {s: {t: [] for t in time_points} for s in sizes}
for p in img_path_list_exp4_cleaned:
    parts = p.split(os.sep)
    t_val = next((t for t in time_points if t in parts), None)
    s_val = next((s for s in sizes if s in parts), None)
    if t_val and s_val:
        data_groups[s_val][t_val].append(p)
max_reps_200 = max(len(data_groups['200'][t]) for t in time_points)
max_reps_300 = max(len(data_groups['300'][t]) for t in time_points)
total_rows = max_reps_200 + max_reps_300

# ==========================================
# 2. PROCESSING
# ==========================================
processed_data = defaultdict(dict)
global_max_val = 0
print(">> Processing all replicates...")
for s in sizes:
    for t in time_points:
        imgs = []
        for p in data_groups[s][t]:
            img_load = analysis.load_img_virtual(path=p, t_sel_idx=0, c_sel_idx=0)
            img_raw, _, img_scale, _ = img_load
            current_res = img_scale[1]
            sf = current_res / REFERENCE_SCALE
            max_proj = np.max(img_raw, axis=0)
            img_rescaled = rescale(max_proj, sf, anti_aliasing=True, preserve_range=True)
            local_99 = np.percentile(img_rescaled, 99.5)
            if local_99 > global_max_val:
                global_max_val = local_99
            imgs.append(img_rescaled)
            print(f"Processed {os.path.basename(p)}: Original Res {current_res:.3f} -> Factor {sf:.2f}")
        processed_data[s][t] = imgs

In [ ]:

plt.rcParams.update({
    'font.family': 'sans-serif',
    'svg.fonttype': 'none',
    'mathtext.fontset': 'stix'
})

# 1. SETUP DIMENSIONS
n_times = len(time_points)
n_sizes = len(sizes)
max_reps = max(len(processed_data[s][t]) for s in sizes for t in time_points)
n_cols_mini = int(np.ceil(np.sqrt(max_reps)))
n_rows_mini = int(np.ceil(max_reps / n_cols_mini))

all_imgs = [im for s in sizes for t in time_points for im in processed_data[s][t]]
canvas_h = max(im.shape[0] for im in all_imgs) + 60
canvas_w = max(im.shape[1] for im in all_imgs) + 60

# Square-ish figure footprint
fig = plt.figure(figsize=(24, 10), facecolor='white')

# Master grid for Time/Size categories
master_gs = gridspec.GridSpec(n_sizes, n_times, figure=fig,
                              wspace=0.15, hspace=0.2,
                              left=0.1, right=0.9, top=0.9, bottom=0.1)

im = None

for s_idx, s in enumerate(sizes):
    for t_idx, t in enumerate(time_points):
        # Mini-grid for replicates
        inner_gs = gridspec.GridSpecFromSubplotSpec(n_rows_mini, n_cols_mini,
                                                    subplot_spec=master_gs[s_idx, t_idx],
                                                    wspace=0.03, hspace=0.03)

        reps = processed_data[s][t]

        for r_idx in range(n_rows_mini * n_cols_mini):
            row_i = r_idx // n_cols_mini
            col_i = r_idx % n_cols_mini
            ax = fig.add_subplot(inner_gs[row_i, col_i])

            if r_idx < len(reps):
                img = reps[r_idx]
                h, w = img.shape
                off_y, off_x = (canvas_h - h) // 2, (canvas_w - w) // 2
                im = ax.imshow(img, cmap='Greys', vmin=0, vmax=global_max_val,
                               origin='upper', extent=[off_x, off_x + w, off_y + h, off_y])

            ax.set_xlim(0, canvas_w)
            ax.set_ylim(canvas_h, 0)
            ax.axis('off')

        # --- REFINED LABELING ---
        # Get the bounding box of the whole master cell to center labels
        cell_ax = fig.add_subplot(master_gs[s_idx, t_idx])
        cell_ax.axis('off')

        # Time Titles (Top of each column)
        if s_idx == 0:
            cell_ax.text(0.5, 1.15, fr"$\mathbf{{{time_labels[t_idx]}}}$",
                         transform=cell_ax.transAxes, fontsize=28, ha='center', va='bottom')

        # Size Labels (Centered vertically on the left of each row)
        if t_idx == 0:
            cell_ax.text(-0.25, 0.5, fr"$\mathbf{{{size_labels[s_idx]}}}$",
                         transform=cell_ax.transAxes, fontsize=26, va='center', ha='right', rotation=90)

# ==========================================
# 4. POLISHED SCALE BAR
# ==========================================
ax_sb = fig.add_axes([0.78, 0.08, 0.12, 0.02])
ax_sb.axis('off')
bar_px = BAR_UM / REFERENCE_SCALE
ax_sb.set_xlim(0, 100)
ax_sb.set_ylim(0, 10)
ax_sb.plot([60, 100], [5, 5], color='black', lw=8, solid_capstyle='butt')
ax_sb.text(80, 8, fr"$\mathbf{{{BAR_UM}}}\ \mathbf{{\mu m}}$",
           ha='center', va='bottom', fontsize=24, fontweight='bold')

# ==========================================
# 5. COLORBAR
# ==========================================
cbar_ax = fig.add_axes([0.92, 0.3, 0.015, 0.4])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.outline.set_edgecolor('black')
cbar.set_label(r"$\mathbf{Fluorescence\ Intensity\ (a.u.)}$", fontsize=20, labelpad=20)
cbar.ax.tick_params(labelsize=14)

# ==========================================
# 6. EXPORT
# ==========================================
final_path = os.path.join(SAVE_DIR, 'all_exp4_max-projections_phase-diagram.png')
plt.savefig(final_path, dpi=500, facecolor='white', bbox_inches='tight')
plt.show()

## Batch Run

In [ ]:
img_list = img_path_list_exp4_cleaned
for i, path in enumerate(img_list):
    print("||||||||||", len(img_list) - i, "left !")
    print(f"=========== [Progress {100 * np.round(i / len(img_list), 2)}%] {path} ===========")
    timepoint = int(path.split("_CLEAN/")[1].split("/")[0])
    nemo_cylindrical_coords.main(img_path=path, overwrite=False, show_figures=False,
                                 voxel_size=2 if timepoint == 72 else 3,
                                 spline_smooth_factor=400, render=False,
                                 force_pca_use=True if timepoint == 72 else False, t_select=t_select, c_select=c_select)
    resdata_dir, resfig_dir = datahandler.create_resdirs(path)
    e_s = datahandler.load_array(name="e_s", folderpath=resdata_dir)
    e_phi = datahandler.load_array(name="e_phi", folderpath=resdata_dir)
    curv_patch_size_all = 50.0
    curv_num_samples = 5000
    nemo_morph_curvature.main(img_path=path, radius=curv_patch_size_all, num_samples=curv_num_samples,
                              show_figures=False, flip_normals=False, custom_basis=(e_s, e_phi), t_select=t_select,
                              c_select=c_select,
                              mesh_name="sampling_mesh", interp_k=10)
    img_unit = analysis.load_img_unit(path)
    for proj_target_distance_all in [5.0, 15.0, 25.0]:
        print(f">>>>>>>> PROJECTION at {proj_target_distance_all}!")
        proj_target_distance_eps = 0.05
        proj_min_dist_all = proj_target_distance_all - proj_target_distance_eps
        proj_max_dist_all = proj_target_distance_all + proj_target_distance_eps
        proj_num_dist_all = 30
        proj_mode_all = "mean"
        layer_label_all = f"sampling_mesh_proj_{proj_min_dist_all}_to_{proj_max_dist_all}_{img_unit}"
        nemo_3_project_img.main(img_path=path, dist_min=proj_min_dist_all, dist_max=proj_max_dist_all,
                                dist_num=proj_num_dist_all, proj_mode=proj_mode_all, render=False,
                                show_figures=False, flip_normals=True, t_select=t_select, c_select=c_select,
                                mesh_name="sampling_mesh")
        dir_extr_patch_size_all = 30
        dir_extr_num = 5000
        nemo_4_extract_nematic.main(img_path=path, layer_label=layer_label_all, render=False, patch_mode="radius",
                                    patch_size=dir_extr_patch_size_all, compute_num=dir_extr_num,
                                    normal_validity_k=20, normal_validity_thresh=0.99, grid_n_2dcurve_analysis=30,
                                    debug_2dcurve_analysis=False, show_figures=False, t_select=t_select,
                                    c_select=c_select)
        cutoff_phi_all = np.pi / 4
        for q_decomp_radius_all in [50.0, 100.0]:
            print(f">>>>>>>> NEMATIC ANALYSIS at {q_decomp_radius_all}!")
            nemo_cylindrical_analyse_nematic.main(img_path=path, layer_label=layer_label_all,
                                                  low_cutoff_phi=-cutoff_phi_all,
                                                  high_cutoff_phi=cutoff_phi_all, profile_bins=20,
                                                  q_decomp_radius=q_decomp_radius_all, show_figures=False,
                                                  t_select=t_select, c_select=c_select)

## Create Batch Datasets

In [ ]:
# Initialize a dictionary of lists (one list per column)
data_dict = {
    "path": [], "Time": [], "Size": [], "GasID": [],
    "s": [], "s_normalised": [], "rho": [], "phi_angle": [],
    "curv_ss": [], "curv_phiphi": [], "curv_sphi": [],
    "curv_midline": [], "Volume": [], "Area": []
}
curv_patch_size_all = 50.0
patch_label = f"r-{curv_patch_size_all}um"

for i, path in enumerate(img_path_list_exp4_cleaned):
    print(f"=========== [Progress {100 * np.round(i / len(img_path_list_exp4_cleaned), 2)}%] {path} ===========")
    t_select = 0
    resdata_dir, resfig_dir = datahandler.create_resdirs(img_path, ct_label=f"t={t_select}_c={c_select}")
    path_parts = path.split(os.sep)
    gas_id = str(path_parts[-1].replace(".tif", ""))
    size_val = int(path_parts[-2])
    t_val = int(path_parts[-3])

    mesh_path = os.path.join(resdata_dir, "sampling_mesh.ply")
    if not os.path.exists(mesh_path):
        continue

    mesh = datahandler.load_mesh(mesh_path)
    mesh_coords = datahandler.load_array(name="mesh_s-rho-phi", folderpath=resdata_dir)

    # --- 1. Load Curvatures ---
    curv_path = os.path.join(resdata_dir, f"sampling_mesh.ply_full_curv_{patch_label}.npz")
    n_verts = len(mesh_coords)

    # Pre-fill arrays with NaNs
    c_ss = np.full(n_verts, np.nan)
    c_pp = np.full(n_verts, np.nan)
    c_sp = np.full(n_verts, np.nan)

    if os.path.exists(curv_path):
        with np.load(curv_path) as curv_data:
            t_mix = curv_data['C_tensors_mixed']
            t_idxs = curv_data['tensor_idxs']
            # Bulk assignment is much faster than .get() in a loop
            c_ss[t_idxs] = t_mix[:, 0, 0]
            c_pp[t_idxs] = t_mix[:, 1, 1]
            c_sp[t_idxs] = t_mix[:, 0, 1]

    # --- 2. Load Midline ---
    midline = datahandler.load_array(name="3d_midline_curve", folderpath=resdata_dir)
    dx = np.gradient(midline, axis=0)
    ddx = np.gradient(dx, axis=0)
    num = np.linalg.norm(np.cross(dx, ddx), axis=1)
    den = np.linalg.norm(dx, axis=1) ** 3
    k_midline = np.divide(num, den, out=np.zeros_like(num), where=den != 0)
    if len(k_midline) > 2:
        k_midline[0], k_midline[-1] = k_midline[1], k_midline[-2]

    # --- 3. Compute Coordinates ---
    s_vals = mesh_coords[:, 0]
    s_max = s_vals.max()
    s_norm = s_vals / s_max if s_max > 0 else s_vals

    # Map midline curvature using vectorized indexing
    mid_idxs = np.clip(s_norm * (len(k_midline) - 1), 0, len(k_midline) - 1).astype(int)
    c_mid = k_midline[mid_idxs]

    # --- 4. Append to Columnar Lists ---
    data_dict["s"].append(s_vals)
    data_dict["s_normalised"].append(s_norm)
    data_dict["rho"].append(mesh_coords[:, 1])
    data_dict["phi_angle"].append(mesh_coords[:, 2])
    data_dict["curv_ss"].append(c_ss)
    data_dict["curv_phiphi"].append(c_pp)
    data_dict["curv_sphi"].append(c_sp)
    data_dict["curv_midline"].append(c_mid)

    # Repeat metadata values to match number of vertices
    data_dict["path"].append(np.repeat(path, n_verts))
    data_dict["Time"].append(np.repeat(t_val, n_verts))
    data_dict["Size"].append(np.repeat(size_val, n_verts))
    data_dict["GasID"].append(np.repeat(gas_id, n_verts))
    data_dict["Volume"].append(np.repeat(mesh.volume, n_verts))
    data_dict["Area"].append(np.repeat(mesh.area, n_verts))

# --- 5. Final Fast Assembly ---
print(">> Concatenating and creating DataFrame...")
final_columns = {k: np.concatenate(v) for k, v in data_dict.items()}
df_morpho_full = pd.DataFrame(final_columns)

print(">> Saving to disk...")
df_morpho_full.to_pickle(os.path.join(gastruloid_batch_output_folder, dataset_morphology_name))

print(f">> Extraction Complete! Saved {len(df_morpho_full)} rows.")

In [ ]:
# Containers for the columnar data
binned_data = {
    "path": [], "Time": [], "Size": [], "GasID": [],
    "s_bin": [], "Q_ss_mean": [], "Q_phiphi_mean": [], "Q_sphi_mean": []
}

full_data = {
    "path": [], "Time": [], "Size": [], "GasID": [],
    "dir_s": [], "dir_rho": [], "dir_phi": [],
    "Q_ss": [], "Q_phiphi": [], "Q_sphi": []
}

for i, path in enumerate(img_path_list_exp4_cleaned):
    print(f"=========== [Progress {100 * np.round(i / len(img_path_list_exp4_cleaned), 2)}%] {path} ===========")

    resdata_dir, resfig_dir = datahandler.create_resdirs(img_path, ct_label=f"t={t_select}_c={c_select}")
    resdata_dir_layer = os.path.join(resdata_dir, layer_label_selected)
    path_parts = path.split(os.sep)
    gas_id = str(path_parts[-1].replace(".tif", ""))
    size_val = int(path_parts[-2])
    t_val = int(path_parts[-3])

    # --- 1. Load Binned Data (Fast, smaller arrays) ---
    s_bins = datahandler.load_array(f"s_bin_centers_cropped_{q_decomp_label_selected}", folderpath=resdata_dir_layer)
    q_ss_m = datahandler.load_array(f"q_ss_mean_cropped_{q_decomp_label_selected}", folderpath=resdata_dir_layer)
    q_pp_m = datahandler.load_array(f"q_phiphi_mean_cropped_{q_decomp_label_selected}", folderpath=resdata_dir_layer)
    q_sp_m = datahandler.load_array(f"q_sphi_mean_cropped_{q_decomp_label_selected}", folderpath=resdata_dir_layer)

    n_bins = len(s_bins)
    binned_data["s_bin"].append(s_bins)
    binned_data["Q_ss_mean"].append(q_ss_m)
    binned_data["Q_phiphi_mean"].append(q_pp_m)
    binned_data["Q_sphi_mean"].append(q_sp_m)

    # Repeat metadata for binned rows
    binned_data["path"].append(np.repeat(path, n_bins))
    binned_data["Time"].append(np.repeat(t_val, n_bins))
    binned_data["Size"].append(np.repeat(size_val, n_bins))
    binned_data["GasID"].append(np.repeat(gas_id, n_bins))

    # --- 2. Load Raw Nematic Points (Heavy arrays) ---
    coords = datahandler.load_array("dir_s-rho-phi_cropped", folderpath=resdata_dir_layer)
    q_data = np.load(os.path.join(resdata_dir_layer, f"q_sphi_full_cropped_{q_decomp_label_selected}.npz"))
    q_tensor = q_data['q_sphi_cropped']

    n_pts = len(coords)
    full_data["dir_s"].append(coords[:, 0])
    full_data["dir_rho"].append(coords[:, 1])
    full_data["dir_phi"].append(coords[:, 2])
    full_data["Q_ss"].append(q_tensor[:, 0, 0])
    full_data["Q_phiphi"].append(q_tensor[:, 1, 1])
    full_data["Q_sphi"].append(q_tensor[:, 0, 1])

    # Repeat metadata for raw points
    full_data["path"].append(np.repeat(path, n_pts))
    full_data["Time"].append(np.repeat(t_val, n_pts))
    full_data["Size"].append(np.repeat(size_val, n_pts))
    full_data["GasID"].append(np.repeat(gas_id, n_pts))

# --- 3. Efficient Assembly ---
print(">> Concatenating Nematic data...")
df_nematic_binned = pd.DataFrame({k: np.concatenate(v) for k, v in binned_data.items()})
df_nematic_full = pd.DataFrame({k: np.concatenate(v) for k, v in full_data.items()})

print(">> Saving Nematic pickles...")
df_nematic_binned.to_pickle(os.path.join(gastruloid_batch_output_folder, dataset_nematic_binned_name))
df_nematic_full.to_pickle(os.path.join(gastruloid_batch_output_folder, dataset_nematic_profile_name))

print(f"Extraction Complete! Binned: {len(df_nematic_binned)} rows, Full: {len(df_nematic_full)} rows.")

## Load Batch Datasets

In [ ]:
df_morpho_full = pd.read_pickle(os.path.join(gastruloid_batch_output_folder, dataset_morphology_name))
target_phi_pos = np.pi / 2
tol = 0.05
target_mid_thickness_range = 100
df_morpho_side = df_morpho_full[
    np.isclose(np.abs(df_morpho_full["phi_angle"]), target_phi_pos, atol=tol)
].copy()
df_morpho_side['mid_point'] = df_morpho_side.groupby(['Time', 'Size', 'GasID'])['s'].transform('max') / 2
is_mid_section = (df_morpho_side['s'] >= df_morpho_side['mid_point'] - target_mid_thickness_range / 2) & \
                 (df_morpho_side['s'] <= df_morpho_side['mid_point'] + target_mid_thickness_range / 2)
summary_base = df_morpho_side.groupby(["Time", "Size", "GasID", "path"]).agg(
    body_length=("s", "max"),
    volume=("Volume", "first"),
    area=("Area", "first"),
    max_rho=("rho", "max"),
    avg_rho=("rho", "mean")
).reset_index()
summary_mid = df_morpho_side[is_mid_section].groupby(["Time", "Size", "GasID"]).agg(
    mid_mean_rho=("rho", "mean"),
    mid_min_rho=("rho", "min")
).reset_index()
df_morpho_summary = summary_base.merge(summary_mid, on=["Time", "Size", "GasID"], how="left")
thickness_map = {
    "max_rho": "Max Body Thickness",
    "avg_rho": "Average Body Thickness",
    "mid_mean_rho": "Mid Body Thickness (Mean)",
    "mid_min_rho": "Mid Body Thickness (Min)"
}
for raw_col, clean_name in thickness_map.items():
    df_morpho_summary[clean_name] = df_morpho_summary[raw_col] * 2
df_morpho_summary = df_morpho_summary.drop(columns=list(thickness_map.keys()))
df_morpho_summary = df_morpho_summary.rename(columns={
    "body_length": "Body Length",
    "volume": "Volume",
    "area": "Area"
})
df_morpho_summary = df_morpho_summary.dropna(subset=["Body Length"])
unique_times = np.sort(df_morpho_summary['Time'].unique())
unique_sizes = np.sort(df_morpho_summary['Size'].unique())

print(f"Summary Cleaned! Metrics available: {list(df_morpho_side.columns)}")
print(f"Total Sample size: {len(df_morpho_side)} gastruloids.")

In [ ]:
pickle_files = [
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/URI_25022026_PR_NEMO/EXP4_filter_membrane/!batch-analysis/df_nematic_sampling_mesh_proj_4.95_to_5.05_um_mean_r-50.0um.pkl',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/URI_25022026_PR_NEMO/EXP4_filter_membrane/!batch-analysis/df_nematic_sampling_mesh_proj_4.95_to_5.05_um_mean_r-100.0um.pkl',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/URI_25022026_PR_NEMO/EXP4_filter_membrane/!batch-analysis/df_nematic_sampling_mesh_proj_14.95_to_15.05_um_mean_r-50.0um.pkl',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/URI_25022026_PR_NEMO/EXP4_filter_membrane/!batch-analysis/df_nematic_sampling_mesh_proj_14.95_to_15.05_um_mean_r-100.0um.pkl',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/URI_25022026_PR_NEMO/EXP4_filter_membrane/!batch-analysis/df_nematic_sampling_mesh_proj_24.95_to_25.05_um_mean_r-50.0um.pkl',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/URI_25022026_PR_NEMO/EXP4_filter_membrane/!batch-analysis/df_nematic_sampling_mesh_proj_24.95_to_25.05_um_mean_r-100.0um.pkl'
]
all_dfs = []
for f_path in pickle_files:
    fname = os.path.basename(f_path)
    proj_match = re.search(r'proj_(.*?)_um', fname)
    r_match = re.search(r'mean_r-(.*?)um', fname)
    proj_label = proj_match.group(1) if proj_match else "unknown"
    r_scale = float(r_match.group(1)) if r_match else 0.0
    df = pd.read_pickle(f_path)
    df['S'] = 2 * np.sqrt(df['Q_ss'] ** 2 + df['Q_sphi'] ** 2)
    df['proj_depth'] = proj_label
    df['averaging_r'] = r_scale
    all_dfs.append(df)
    print(f"Loaded: {proj_label} um | r={r_scale} um")

df_nematic_full = pd.concat(all_dfs, ignore_index=True)
print(f"\nFinal Master DF shape: {df_nematic_full.shape}")
print(f"Unique Projections: {df_nematic_full['proj_depth'].unique()}")
print(f"Unique R-scales: {df_nematic_full['averaging_r'].unique()}")

## Visualise Batch Datasets

In [ ]:
# --- RESTORE FONT TYPE ---
plt.rcParams.update({
    'font.family': 'STIXGeneral',
    'font.serif': ['DejaVu Serif', 'Times New Roman'],
    'mathtext.fontset': 'stix',  # Computer Modern (standard for journals)
    'text.usetex': False  # Ensures we use Matplotlib's internal mathtext
})
sns.set_theme(style="ticks")


def plot_combined_phase_diagram(df_nematic, df_morpho, selected_layer, selected_r, save_path,
                                nem_comp='S', mor_comp='curv_midline',
                                nem_ylims=None, mor_ylims=None):
    # 1. Filter and Prep Data
    df_nem = df_nematic[(df_nematic["proj_depth"] == selected_layer) &
                        (df_nematic["averaging_r"] == selected_r)].copy()
    df_mor = df_morpho.copy()

    for df, col_name in [(df_nem, 's_plot'), (df_mor, 's_mor_plot')]:
        if col_name not in df.columns:
            source = 'dir_s' if 'dir_s' in df.columns else 's_normalised'
            df[col_name] = df.groupby("path")[source].transform(lambda x: x / x.max())

    unique_times = np.sort(df_nem['Time'].unique())
    unique_sizes = np.sort(df_nem['Size'].unique())
    n_rows, n_cols = len(unique_sizes), len(unique_times)

    cmap_base = plt.get_cmap('coolwarm')
    time_colors = {t: cmap_base(i / (len(unique_times) - 1)) for i, t in enumerate(unique_times)}

    def get_robust_limits(df, comp):
        vmin, vmax = df[comp].quantile(0.01), df[comp].quantile(0.99)
        margin = (vmax - vmin) * 0.15
        return vmin - margin, vmax + margin

    if nem_ylims is None: nem_ylims = get_robust_limits(df_nem, nem_comp)
    if mor_ylims is None: mor_ylims = get_robust_limits(df_mor, mor_comp)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.7 * n_cols, 4 * n_rows), sharex=True)
    axes = np.atleast_2d(axes)
    x_eval = np.linspace(-0.01, 1.01, 200)

    for r, sz_val in enumerate(unique_sizes):
        for c, t_val in enumerate(unique_times):
            ax_nem = axes[r, c]
            ax_mor = ax_nem.twinx()
            color_t = time_colors[t_val]

            def plot_lifestyle(ax, data, comp, x_col, main_color, is_dashed=False, alpha_indiv=0.25):
                subset = data[(data['Time'] == t_val) & (data['Size'] == sz_val)]
                profiles = []
                for _, g in subset.groupby("path"):
                    g = g.dropna(subset=[x_col, comp]).sort_values(x_col)
                    if len(g) < 15: continue
                    y_smooth = gaussian_filter1d(np.interp(x_eval, g[x_col], g[comp]), sigma=6)
                    profiles.append(y_smooth)
                    ax.plot(x_eval, y_smooth, color=main_color, alpha=alpha_indiv, lw=0.8, zorder=1)

                if profiles:
                    matrix = np.array(profiles)
                    mean, sem = np.nanmean(matrix, axis=0), np.nanstd(matrix, axis=0) / np.sqrt(len(profiles))
                    ax.fill_between(x_eval, mean - sem, mean + sem, color=main_color, alpha=0.2, zorder=2)
                    ls = '--' if is_dashed else '-'
                    ax.plot(x_eval, mean, color='black', lw=2.5, ls=ls, zorder=3)
                    ax.plot(x_eval, mean, color=main_color, lw=1.2, ls=ls, zorder=4)
                return len(profiles)

            n_nem = plot_lifestyle(ax_nem, df_nem, nem_comp, 's_plot', color_t, is_dashed=False, alpha_indiv=0.25)
            _ = plot_lifestyle(ax_mor, df_mor, mor_comp, 's_mor_plot', color_t, is_dashed=True, alpha_indiv=0.25)

            ax_nem.set_ylim(nem_ylims)
            ax_mor.set_ylim(mor_ylims)
            ax_nem.yaxis.set_major_locator(MaxNLocator(4))
            ax_mor.yaxis.set_major_locator(MaxNLocator(4))

            sns.despine(ax=ax_nem, left=False, right=True, top=True, bottom=False, offset=5, trim=False)
            sns.despine(ax=ax_mor, left=True, right=False, top=True, bottom=True, offset=5, trim=False)
            ax_mor.spines['right'].set_visible(True)

            # --- FIX: UNIFORM FONT WEIGHT IN TITLES ---
            if r == 0:
                ax_nem.set_title(fr"$t = {t_val}$ h", fontsize=16, pad=15)

            if c == n_cols - 1:
                # Same for size labels
                ax_mor.set_ylabel(fr"Size = {sz_val}", fontsize=12, labelpad=5)

            ax_nem.text(0.95, 0.05, f"$n={n_nem}$", transform=ax_nem.transAxes,
                        ha='right', fontsize=11, fontweight='bold',
                        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))

    # Consistent LaTeX for global labels
    fig.supxlabel(r"Normalized Arc Length $s/s_{max}$", fontsize=18, y=0.02)

    fig.text(0.01, 0.5, f"Nematic Component: {nem_comp} (solid line)",
             va='center', rotation='vertical', fontsize=18)

    fig.text(0.99, 0.5, f"Morphology: {mor_comp} (dashed line)",
             va='center', rotation=270, fontsize=18, color='#333333')

    plt.tight_layout(rect=[0.03, 0.05, 0.99, 0.97])

    save_name = f"combined_{nem_comp}_{selected_layer}_r=-{selected_r}_VS_{mor_comp}.pdf"
    plt.savefig(os.path.join(save_path, save_name), dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()


# --- RESTORE FONT TYPE ---
plt.rcParams.update({
    'font.family': 'STIXGeneral',
    'mathtext.fontset': 'stix',
    'text.usetex': False
})
sns.set_theme(style="ticks")


def plot_morphology_dual_phase(df, save_path, left_comp='rho', right_comp='curv_midline'):
    # Pre-calculate normalized arc length if missing
    if 's_norm' not in df.columns:
        df['s_norm'] = df.groupby("path")['s'].transform(lambda x: x / x.max())

    unique_times = np.sort(df['Time'].unique())
    unique_sizes = np.sort(df['Size'].unique())
    n_rows, n_cols = len(unique_sizes), len(unique_times)

    cmap_base = plt.get_cmap('coolwarm')
    time_colors = {t: cmap_base(i / (len(unique_times) - 1)) for i, t in enumerate(unique_times)}

    def get_robust_limits(comp):
        vmin, vmax = df[comp].quantile(0.01), df[comp].quantile(0.99)
        margin = (vmax - vmin) * 0.2
        return (vmin - margin, vmax + margin)

    left_ylims = get_robust_limits(left_comp)
    right_ylims = get_robust_limits(right_comp)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.7 * n_cols, 4 * n_rows), sharex=True)
    axes = np.atleast_2d(axes)
    x_eval = np.linspace(-0.01, 1.01, 200)

    for r, sz_val in enumerate(unique_sizes):
        for c, t_val in enumerate(unique_times):
            ax_left = axes[r, c]
            ax_right = ax_left.twinx()
            color_t = time_colors[t_val]

            def plot_trends(ax, comp, is_dashed=False, alpha_indiv=0.2):
                subset = df[(df['Time'] == t_val) & (df['Size'] == sz_val)]
                profiles = []
                for _, g in subset.groupby("path"):
                    g = g.dropna(subset=['s_norm', comp]).sort_values('s_norm')
                    if len(g) < 15: continue
                    y_smooth = gaussian_filter1d(np.interp(x_eval, g['s_norm'], g[comp]), sigma=6)
                    profiles.append(y_smooth)
                    ax.plot(x_eval, y_smooth, color=color_t, alpha=alpha_indiv, lw=0.8, zorder=1)

                if profiles:
                    matrix = np.array(profiles)
                    mean = np.nanmean(matrix, axis=0)
                    sem = np.nanstd(matrix, axis=0) / np.sqrt(len(profiles))
                    ax.fill_between(x_eval, mean - sem, mean + sem, color=color_t, alpha=0.15, zorder=2)
                    ls = '--' if is_dashed else '-'
                    ax.plot(x_eval, mean, color='black', lw=2.2, ls=ls, zorder=3)
                    ax.plot(x_eval, mean, color=color_t, lw=1.2, ls=ls, zorder=4)
                return len(profiles)

            n_samples = plot_trends(ax_left, left_comp, is_dashed=False)
            _ = plot_trends(ax_right, right_comp, is_dashed=True)

            ax_left.set_ylim(left_ylims)
            ax_right.set_ylim(right_ylims)
            ax_left.yaxis.set_major_locator(MaxNLocator(4))
            ax_right.yaxis.set_major_locator(MaxNLocator(4))

            sns.despine(ax=ax_left, left=False, right=True, top=True, bottom=False, offset=5, trim=False)
            sns.despine(ax=ax_right, left=True, right=False, top=True, bottom=True, offset=5, trim=False)
            ax_right.spines['right'].set_visible(True)

            if r == 0:
                ax_left.set_title(fr"$t = {t_val}$ h", fontsize=16, pad=15)
            if c == n_cols - 1:
                ax_right.set_ylabel(fr"Size = {sz_val}", fontsize=12, labelpad=5)
            ax_left.text(0.95, 0.05, f"$n={n_samples}$", transform=ax_left.transAxes,
                         ha='right', fontsize=10, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

    fig.supxlabel(r"Normalized Arc Length, $s/s_{max}$", fontsize=18, y=0.02)
    fig.text(0.01, 0.5, fr"{left_comp} (solid line)", va='center', rotation='vertical', fontsize=18)
    fig.text(0.99, 0.5, fr"{right_comp} (dashed line)", va='center', rotation=270, fontsize=18)

    plt.tight_layout(rect=[0.03, 0.05, 0.99, 0.97])
    save_name = f"morpho_comparison_{left_comp}_vs_{right_comp}.pdf"
    plt.savefig(os.path.join(save_path, save_name), dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
# Updated Lists based on your provided keys
nematic_list = ['S', 'Q_ss', 'Q_phiphi', 'Q_sphi']
morpho_list = ['rho', 'curv_midline', 'curv_ss', 'curv_phiphi', 'curv_sphi']
#
# nematic_list = ["S"]
# morpho_list = ["phi_angle", "s"]
# Execution loop for all combinations
for nem_key in nematic_list:
    # Scale S to [0, 1]; use robust scaling for Q components
    n_ylims = [0, 1] if nem_key == 'S' else None

    for mor_key in morpho_list:
        try:
            print(f"Generating diagram: Nematic({nem_key}) + Morpho({mor_key})")
            plot_combined_phase_diagram(
                df_nematic=df_nematic_full,
                df_morpho=df_morpho_side,
                selected_layer=selected_layer_label,
                selected_r=selected_averaging_r,
                save_path=morpho_nematic_batch_analysis_path,
                nem_comp=nem_key,
                mor_comp=mor_key,
                nem_ylims=n_ylims
            )
        except Exception as e:
            print(f"Failed to plot {nem_key} vs {mor_key}: {e}")

# --- EXECUTION BLOCK ---
morpho_targets = ['curv_midline', 'curv_ss', 'curv_phiphi', 'curv_sphi']

for target in morpho_targets:
    try:
        plot_morphology_dual_phase(
            df=df_morpho_side,
            save_path=morpho_nematic_batch_analysis_path,
            left_comp='rho',
            right_comp=target
        )
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
def plot_nematic_multiscale_phase_diagram(df_raw, save_path, component='S', ydashed=0.0):
    df = df_raw.copy()

    # 1. Component-specific logic (S is always 0 to 1)
    if component == 'S':
        ylims = (0.0, 1.0)
        comp_tex = r"S"
    else:
        ylims = (-0.55, 0.55)
        comp_tex = fr"Q_{{{component.split('_')[-1]}}}"

    # Ensure s_plot exists
    df['s_plot'] = df.groupby(["path", "proj_depth", "averaging_r"])['dir_s'].transform(lambda x: x / x.max())

    unique_times = np.sort(df['Time'].unique())
    unique_sizes = np.sort(df['Size'].unique())

    # Sort depths numerically by the first number in the string (e.g., "4.95" from "4.95_to_5.05")
    unique_depths = sorted(df['proj_depth'].unique(), key=lambda x: float(x.split('_')[0]))
    unique_scales = np.sort(df['averaging_r'].unique())

    # 2. Mappings
    # Depth -> Magma colormap (reversed looks better for 'increasing depth')
    depth_colors = plt.cm.rainbow(np.linspace(0.1, 0.85, len(unique_depths)))
    depth_to_color = {d: depth_colors[i] for i, d in enumerate(unique_depths)}

    # Scale -> Linestyle
    styles = ['-', '--', ':', '-.']
    scale_to_style = {r: styles[i % len(styles)] for i, r in enumerate(unique_scales)}

    n_rows = len(unique_sizes)
    n_cols = len(unique_times)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4.5 * n_rows),
                             sharex=True, sharey=True, facecolor='white')

    if n_rows == 1: axes = np.atleast_2d(axes)
    if n_cols == 1: axes = np.atleast_2d(axes).T

    x_eval = np.linspace(0, 1, 200)

    for r, sz_val in enumerate(unique_sizes):
        for c, t_val in enumerate(unique_times):
            ax = axes[r, c]
            cell_data = df[(df['Time'] == t_val) & (df['Size'] == sz_val)]

            if cell_data.empty:
                ax.axis('off')
                continue

            for d_val in unique_depths:
                for r_scale in unique_scales:
                    subset = cell_data[(cell_data['proj_depth'] == d_val) &
                                       (cell_data['averaging_r'] == r_scale)]

                    if subset.empty: continue

                    color = depth_to_color[d_val]
                    linestyle = scale_to_style[r_scale]

                    all_interp = []
                    for _, gast_data in subset.groupby("path"):
                        gast_data = gast_data.dropna(subset=['s_plot', component]).sort_values('s_plot')
                        if len(gast_data) < 10: continue

                        y_interp = np.interp(x_eval, gast_data['s_plot'], gast_data[component])
                        y_smooth = gaussian_filter1d(y_interp, sigma=4)
                        all_interp.append(y_smooth)

                    if not all_interp: continue

                    matrix = np.array(all_interp)
                    mean_line = np.nanmean(matrix, axis=0)
                    sem_line = np.nanstd(matrix, axis=0) / np.sqrt(len(all_interp))

                    ax.fill_between(x_eval, mean_line - sem_line, mean_line + sem_line,
                                    color=color, alpha=0.15, zorder=2)
                    ax.plot(x_eval, mean_line, color=color, linewidth=2.5,
                            linestyle=linestyle, zorder=3)

            ax.set_ylim(ylims)
            ax.axhline(ydashed, color='black', linewidth=1, alpha=0.2, linestyle='--')
            sns.despine(ax=ax, offset=5, trim=True)

            if r == 0:
                ax.set_title(fr"$\mathbf{{{t_val}\ hps}}$", fontsize=18, pad=15)
            if c == n_cols - 1:
                ax.set_ylabel(fr"$\mathbf{{{sz_val}\ cells}}$", fontsize=16, fontweight='bold', labelpad=20)
                ax.yaxis.set_label_position("right")

    fig.supxlabel(r"Normalized Arc Length, $s/s_{max}$", fontsize=20, y=0.02)
    fig.supylabel(fr"Nematic Component: ${comp_tex}$", fontsize=20, x=0.02)

    # Global Legend
    legend_elements = []
    for d in unique_depths:
        legend_elements.append(Line2D([0], [0], color=depth_to_color[d], lw=4, label=fr"Projection depth: {d} $\mu$m"))
    for r_sc in unique_scales:
        legend_elements.append(
            Line2D([0], [0], color='black', lw=2, linestyle=scale_to_style[r_sc],
                   label=fr"Averaging radius: {r_sc} $\mu$m"))

    fig.legend(handles=legend_elements, loc='lower center', ncol=len(unique_depths),
               bbox_to_anchor=(0.5, -0.08), fontsize=14, frameon=False)

    plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95])

    save_name = f"nematic_{component}_vs_projdepth_avgradius.png"
    plt.savefig(os.path.join(save_path, save_name), dpi=300, bbox_inches='tight')
    plt.show()


# Execution
plot_nematic_multiscale_phase_diagram(df_nematic_full[df_nematic_full["averaging_r"] == 100.0],
                                      nematic_batch_analysis_path, component='S')

In [ ]:

# --- 1. SETUP COLORMAP ---
unique_times = np.sort(df_morpho_side['Time'].unique())
cmap_ref = plt.get_cmap('coolwarm')
colors = cmap_ref(np.linspace(0, 1, len(unique_times)))
time_to_color = {t: colors[i] for i, t in enumerate(unique_times)}


# --- 2. DEFINE PLOTTING FUNCTION ---
def plot_morpho_curvature_analysis(df_raw, save_path, component='curv_ss'):
    df = df_raw.copy()

    # Calculate mean_surface on the fly
    if component == 'mean_surface' and 'mean_surface' not in df.columns:
        df['mean_surface'] = (df['curv_ss'] + df['curv_phiphi']) / 2

    x_col = 's_normalised'
    unique_times_plot = np.sort(df['Time'].unique())
    unique_sizes_plot = np.sort(df['Size'].unique())

    # --- AUTO-LIMITS CALCULATION ---
    # Find global min/max for this specific component to fix the "shit" axes
    data_min = df[component].quantile(0.01)  # Use 1st percentile to avoid outlier stretch
    data_max = df[component].quantile(0.99)  # Use 99th percentile
    margin = (data_max - data_min) * 0.15
    ylims = (data_min - margin, data_max + margin)

    n_rows, n_cols = len(unique_sizes_plot), len(unique_times_plot)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 4 * n_rows),
                             sharex=True, sharey=True)

    axes = np.atleast_2d(axes)
    if n_cols == 1 and n_rows > 1: axes = axes.T

    x_eval = np.linspace(0, 1, 200)

    for r, sz_val in enumerate(unique_sizes_plot):
        for c, t_val in enumerate(unique_times_plot):
            ax = axes[r, c]
            cell_data = df[(df['Time'] == t_val) & (df['Size'] == sz_val)].copy()

            if cell_data.empty:
                ax.axis('off')
                continue

            all_interp = []
            color = time_to_color.get(t_val, 'gray')

            for _, gast_data in cell_data.groupby("path"):
                gast_data = gast_data.dropna(subset=[x_col, component]).sort_values(x_col)
                if len(gast_data) < 15: continue

                y_interp = np.interp(x_eval, gast_data[x_col], gast_data[component])
                y_smooth = gaussian_filter1d(y_interp, sigma=4)
                all_interp.append(y_smooth)
                ax.plot(x_eval, y_smooth, color=color, alpha=0.3, linewidth=1.0, zorder=1)

            if all_interp:
                mat = np.array(all_interp)
                mean, sem = np.nanmean(mat, axis=0), np.nanstd(mat, axis=0) / np.sqrt(len(all_interp))
                ax.fill_between(x_eval, mean - sem, mean + sem, color=color, alpha=0.2, zorder=2)
                ax.plot(x_eval, mean, color='black', linewidth=2.0, zorder=3)

                ax.text(0.95, 0.95, f"$n={len(all_interp)}$", transform=ax.transAxes,
                        ha='right', va='top', fontsize=10, fontweight='bold',
                        bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))

            # Apply robust limits and clean up ticks
            ax.set_ylim(ylims)
            ax.yaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
            ax.tick_params(axis='both', which='major', labelsize=12)

            sns.despine(ax=ax, offset=5, trim=True)

            if r == 0: ax.set_title(fr"$t = {t_val}$ h", fontsize=16, pad=15)
            if c == n_cols - 1:
                ax.set_ylabel(f"Size {sz_val}", fontsize=14, fontweight='bold', labelpad=20)
                ax.yaxis.set_label_position("right")

    comp_tex = component.replace('_', r'\_')
    fig.supxlabel(r"Normalized Arc Length, $s/s_{max}$", fontsize=18, y=0.02)
    fig.supylabel(fr"Curvature Component: $\mathbf{{{comp_tex}}}$", fontsize=18, x=0.02)

    plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95])
    plt.savefig(os.path.join(save_path, f"morpho_grid_{component}.png"), dpi=300, facecolor='white',
                bbox_inches='tight')
    plt.show()


# --- 3. EXECUTION ---
for comp in ['mean_surface', 'curv_ss', 'curv_phiphi', 'curv_sphi', 'curv_midline']:
    try:
        plot_morpho_curvature_analysis(df_morpho_side, morphology_batch_analysis_path, component=comp)
    except Exception as e:
        print(f"Failed {comp}: {e}")

In [ ]:



def plot_3d_professional_reconstruction(df, save_path):
    """
    Consolidated 3D Lofting:
    - Row 1: Size 200 | Row 2: Size 300
    - Full Frenet-Serret bending integration
    - Publication-quality lighting and elevated 3D view
    """
    unique_times = np.sort(df['Time'].unique())
    # Explicitly define target sizes to match the requested row structure
    target_sizes = [200, 300]

    n_rows = len(target_sizes)
    n_cols = len(unique_times)

    fig = plt.figure(figsize=(6 * n_cols, 7 * n_rows))
    fig.suptitle(r"$\mathbf{3D\ Morphological\ Lofting\ Comparison}$", fontsize=28, y=0.98)

    s_fine = np.linspace(0, 1, 200)
    phi = np.linspace(0, 2 * np.pi, 80)
    ls = mcolors.LightSource(azdeg=315, altdeg=45)

    for r, sz_val in enumerate(target_sizes):
        size_data = df[df['Size'] == sz_val]

        for c, t_val in enumerate(unique_times):
            # FIXED: Ensure idx is a pure integer
            idx = int(r * n_cols + c + 1)
            ax = fig.add_subplot(n_rows, n_cols, idx, projection='3d')

            group = size_data[size_data['Time'] == t_val]
            if group.empty:
                ax.axis('off')
                continue

            # --- 1. PARAMETER INTERPOLATION ---
            rho_list, k_list = [], []
            for p in group['path'].unique():
                gast = group[group['path'] == p].sort_values('s_normalised')
                if len(gast) < 15: continue
                rho_list.append(np.interp(s_fine, gast['s_normalised'], gast['rho']))
                k_list.append(np.interp(s_fine, gast['s_normalised'], gast['curv_midline']))

            if not rho_list:
                ax.axis('off')
                continue

            # Smooth and spline average profiles
            cs_rho = CubicSpline(s_fine, gaussian_filter1d(np.nanmean(rho_list, axis=0), sigma=2))
            cs_k = CubicSpline(s_fine, gaussian_filter1d(np.nanmean(k_list, axis=0), sigma=2))

            avg_s_max = group.groupby("path")['s'].max().mean()
            ds = (s_fine[1] - s_fine[0]) * avg_s_max

            # --- 2. BENDING & LOFTING (Frenet-Serret) ---
            alpha = np.cumsum(cs_k(s_fine) * ds)
            mx, my = np.cumsum(np.cos(alpha) * ds), np.cumsum(np.sin(alpha) * ds)

            X, Y, Z = np.zeros((len(s_fine), len(phi))), np.zeros((len(s_fine), len(phi))), np.zeros(
                (len(s_fine), len(phi)))
            for j in range(len(s_fine)):
                nx, ny = -np.sin(alpha[j]), np.cos(alpha[j])
                r_val = cs_rho(s_fine[j])
                X[j, :] = mx[j] + r_val * nx * np.cos(phi)
                Y[j, :] = my[j] + r_val * ny * np.cos(phi)
                Z[j, :] = r_val * np.sin(phi)

            # --- 3. RENDERING ---
            color = time_to_color.get(t_val, 'gray')
            ax.plot_surface(X, Y, Z, color=color, alpha=0.9, shade=True,
                            antialiased=True, rcount=100, ccount=100, lightsource=ls)

            # --- 4. STYLING ---
            if c == 0:
                ax.text2D(-0.15, 0.5, f"Size {sz_val}", transform=ax.transAxes,
                          fontsize=22, fontweight='bold', rotation=90, va='center')

            ax.set_title(f"$t = {t_val}$ h", fontsize=18, pad=0)

            # Robust Dynamic Scaling
            ax.set_xlim(0, avg_s_max * 1.2)
            limit = avg_s_max * 0.6
            ax.set_ylim(-limit, limit)
            ax.set_zlim(-limit, limit)

            ax.set_box_aspect((1, 1, 1))
            ax.view_init(elev=35, azim=-55)
            ax.set_axis_off()

    plt.tight_layout(rect=[0.05, 0.03, 0.95, 0.93])
    save_path_full = os.path.join(save_path, "morpho_3D_grid_reconstruction.png")
    plt.savefig(save_path_full, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()


# Execution
plot_3d_professional_reconstruction(df_morpho_side, morphology_batch_analysis_path)

In [ ]:
sns.set_theme(style="ticks")
unique_sizes = np.sort(df_morpho_summary['Size'].unique())
neutral_palette = ["#bdbdbd", "#636363"]
unique_times = np.sort(df_morpho_summary['Time'].unique())
all_hours = np.arange(unique_times.min(), unique_times.max() + 8, 8)


def plot_experiment_stats(df, save_path, show_fig=True):
    """
    Generates and saves a countplot of gastruloid sample sizes across
    different time points and seeding sizes.
    """
    # ==== Pre-process Data ====
    # Ensure we only count unique gastruloids based on their file path
    df_unique = df.drop_duplicates(subset=["path"])
    unique_sizes = sorted(df_unique["Size"].unique())

    # ==== Styling ====
    neutral_colors = ["#bdbdbd", "#636363"]
    fig, ax = plt.subplots(figsize=(8, 5))

    # ==== Plotting ====
    sns.countplot(
        data=df_unique,
        x="Time",
        hue="Size",
        palette=neutral_colors,
        hue_order=unique_sizes,
        edgecolor="black",
        linewidth=1.0,
        alpha=0.9,
        ax=ax
    )

    # ==== Annotate Bar Heights (n=...) ====
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(
                f'$n={int(height)}$',
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='bottom',
                xytext=(0, 5),
                textcoords='offset points',
                fontsize=10,
                weight='medium'
            )

    # ==== Final Formatting ====
    ax.set_title(r"$\mathbf{Experiment\ Statistics}$", pad=20, fontsize=14)
    ax.set_ylabel(r"Number of gastruloids, $n$", fontsize=12)
    ax.set_xlabel(r"Time post-seeding, $t$ (h)", fontsize=12)
    ax.legend(title=r"Seeding Size (cells)", frameon=False, loc='upper left', bbox_to_anchor=(1, 1))

    # Give some headroom for annotations
    ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
    sns.despine(trim=True, offset=10)
    plt.tight_layout()

    # ==== Save and Show ====
    save_filename = os.path.join(save_path, "experiment_sample_size_stats.pdf")
    plt.savefig(save_filename, bbox_inches='tight', dpi=200)
    print(f">> Experiment stats saved to: {save_filename}")

    if show_fig:
        plt.show()
    else:
        plt.close(fig)

    return fig, ax


def get_stars(p):
    """Standard scientific notation for p-values."""
    if p < 0.0001: return "****"
    if p < 0.001:  return "***"
    if p < 0.01:   return "**"
    if p < 0.05:   return "*"
    return "ns"


def plot_advanced_morphometrics(df_plot, save_path, normalise=False):
    metrics = ["Body Length", "Max Body Thickness",
               "Mid Body Thickness (Mean)", "Mid Body Thickness (Min)"]

    ylabels = [fr"Rel. {m} (norm. to $t_{{min}}$)" for m in metrics] if normalise else [
        r"Body Length ($\mu m$)", r"Max Body Thickness ($\mu m$)",
        fr"Mean Mid Body Thickness ($\mu m$)",
        fr"Min Mid Body Thickness ($\mu m$)"
    ]

    suffix = "_normalised" if normalise else ""
    title_prefix = "Normalised" if normalise else "Gastruloid"

    for metric, ylabel in zip(metrics, ylabels):
        temp_df = df_plot.copy()

        # --- NORMALIZATION ---
        if normalise:
            t_min = unique_times[0]
            for sz in unique_sizes:
                base_val = temp_df[(temp_df["Time"] == t_min) & (temp_df["Size"] == sz)][metric].mean()
                if not np.isnan(base_val) and base_val != 0:
                    idx = temp_df["Size"] == sz
                    temp_df.loc[idx, metric] = temp_df.loc[idx, metric] / base_val

        # --- CATEGORICAL GRID LOGIC ---
        # This forces the x-axis to respect the 8h steps even if data is missing
        temp_df['Time'] = pd.Categorical(temp_df['Time'], categories=all_hours, ordered=True)

        fig, ax = plt.subplots(figsize=(10, 6))

        # 1. Base Box/Strip Plots
        sns.boxplot(data=temp_df, x="Time", y=metric, hue="Size", palette=neutral_palette,
                    ax=ax, fliersize=0, boxprops={'alpha': 0.3}, zorder=1, width=0.7)

        sns.stripplot(data=temp_df, x="Time", y=metric, hue="Size", palette=neutral_palette,
                      ax=ax, dodge=True, alpha=0.4, size=5, edgecolor='black',
                      linewidth=0.3, zorder=2)

        # 2. Manual Mean Lines (connecting across the 8h grid gaps)
        for idx, sz in enumerate(unique_sizes):
            means = []
            x_indices = []
            for t in unique_times:
                m_val = temp_df[(temp_df["Time"] == t) & (temp_df["Size"] == sz)][metric].mean()
                if not np.isnan(m_val):
                    means.append(m_val)
                    # align with the dodge: center of categorical index + offset
                    x_pos = list(all_hours).index(t) + (idx - 0.5) * 0.4
                    x_indices.append(x_pos)

            ax.plot(x_indices, means, color=neutral_palette[idx], linewidth=2, zorder=3, alpha=0.8)
            ax.scatter(x_indices, means, color=neutral_palette[idx], marker='s', s=40,
                       edgecolor='black', linewidth=0.5, zorder=4)

        # 3. Statistical Testing & Annotation Loop
        y_max = temp_df[metric].max()
        y_min = temp_df[metric].min()
        if np.isnan(y_max) or np.isnan(y_min): continue
        y_range = y_max - y_min

        for t in unique_times:
            i = list(all_hours).index(t)
            group1 = temp_df[(temp_df["Time"] == t) & (temp_df["Size"] == unique_sizes[0])][metric].dropna()
            group2 = temp_df[(temp_df["Time"] == t) & (temp_df["Size"] == unique_sizes[1])][metric].dropna()

            n1, n2 = len(group1), len(group2)
            n_y_pos = y_min - (y_range * 0.12)

            # Label n-counts
            ax.text(i - 0.2, n_y_pos, f"$n={n1}$", ha='center', va='top', fontsize=9,
                    color=neutral_palette[0], fontweight='bold')
            ax.text(i + 0.2, n_y_pos, f"$n={n2}$", ha='center', va='top', fontsize=9,
                    color=neutral_palette[1], fontweight='bold')

            # Stars & Brackets
            if n1 > 2 and n2 > 2:
                _, p = mannwhitneyu(group1, group2)
                stars = get_stars(p)
                local_max = max(group1.max(), group2.max())
                h = y_range * 0.05
                y_bracket = local_max + h
                ax.plot([i - 0.2, i - 0.2, i + 0.2, i + 0.2],
                        [y_bracket, y_bracket + h / 2, y_bracket + h / 2, y_bracket],
                        lw=1.2, c='k')
                ax.text(i, y_bracket + h / 2, stars, ha='center', va='bottom', color='k', fontsize=11)

        # 4. Aesthetics
        ax.set_title(fr"$\text{{{title_prefix}\ {metric}}}$", pad=25, fontsize=15)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_xlabel(r"Time post-seeding, $t$ (h)", fontsize=12)

        # Only label the ticks that actually have data
        ax.set_xticks(range(len(all_hours)))
        ax.set_xticklabels([str(h) if h in unique_times else "" for h in all_hours])

        if normalise:
            ax.axhline(1, color='black', linestyle='--', alpha=0.3, linewidth=1, zorder=0)

        # Legend using manual Line2D to match the point/line style
        custom_lines = [Line2D([0], [0], color=neutral_palette[0], lw=2, marker='s', markersize=8),
                        Line2D([0], [0], color=neutral_palette[1], lw=2, marker='s', markersize=8)]
        ax.legend(custom_lines, [f"Size {unique_sizes[0]}", f"Size {unique_sizes[1]}"],
                  title=r"$\text{Seeding Size (cells)}$", frameon=False, loc='upper left', bbox_to_anchor=(1, 1))

        ax.set_ylim(y_min - (y_range * 0.2), y_max + (y_range * 0.3))
        sns.despine(offset=10, trim=True)
        plt.tight_layout()

        save_name = f"morpho_stats_{metric.replace(' ', '_')}{suffix}.pdf"
        plt.savefig(os.path.join(save_path, save_name), bbox_inches='tight', dpi=200, facecolor='white')
        plt.show()
        plt.close(fig)


# --- 1. SETUP ---
unique_times = np.sort(df_morpho_side['Time'].unique())
unique_sizes = np.sort(df_morpho_side['Size'].unique())

# Correct Modern Colormap call
cmap_ref = plt.get_cmap('coolwarm')
colors = cmap_ref(np.linspace(0, 1, len(unique_times)))
time_to_color = {t: colors[i] for i, t in enumerate(unique_times)}

line_styles = ['-', '--', ':', '-.']
size_to_style = {sz: line_styles[i % len(line_styles)] for i, sz in enumerate(unique_sizes)}


def add_custom_legend(fig, axes, unique_sizes, unique_times, size_to_style, time_to_color):
    """Adds the seeding size legend and the time colorbar."""
    # 1. Seeding Size Legend
    print(f"Adding custom legend for {len(axes)}")
    size_handles = [Line2D([0], [0], color='gray', linestyle=size_to_style[sz],
                           label=f"Size {sz}") for sz in unique_sizes]
    fig.legend(handles=size_handles, loc='upper right', bbox_to_anchor=(0.98, 0.85),
               title=r"$\text{Seeding Size}$", frameon=False)

    # 2. Time Colorbar
    colors_list = [time_to_color[t] for t in unique_times]
    sm = cm.ScalarMappable(cmap=ListedColormap(colors_list),
                           norm=BoundaryNorm(np.arange(len(unique_times) + 1) - 0.5, len(unique_times)))

    cbar_ax = fig.add_axes([0.90, 0.25, 0.02, 0.5])
    cbar = fig.colorbar(sm, cax=cbar_ax, ticks=np.arange(len(unique_times)))
    cbar.ax.set_yticklabels([str(int(t)) for t in unique_times])
    cbar.set_label(r"Time post-seeding, $t$ (h)")


def plot_individual_thickness_profiles_split(df, save_path):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    plt.subplots_adjust(right=0.88, hspace=0.15)
    size_to_ax = {unique_sizes[0]: ax1, unique_sizes[1]: ax2}

    for path_id, group in df.groupby("path"):
        sz = group["Size"].iloc[0]
        t = group["Time"].iloc[0]
        ax = size_to_ax[sz]
        trace = group[['s', 'rho']].sort_values("s")

        ax.plot(trace["s"], trace["rho"], color=time_to_color[t],
                linestyle=size_to_style[sz], linewidth=1.0, alpha=0.4, zorder=2)

    ax1.set_title(fr"$\mathbf{{Individual\ Traces\ -\ {unique_sizes[0]}\ cells}}$", fontsize=14)
    ax2.set_title(fr"$\mathbf{{Individual\ Traces\ -\ {unique_sizes[1]}\ cells}}$", fontsize=14)
    ax2.set_xlabel(r"Arc Length, $s$ ($\mu m$)", fontsize=12)

    for ax in [ax1, ax2]:
        ax.set_ylabel(r"Radius, $\rho$ ($\mu m$)", fontsize=12)
        ax.set_ylim(bottom=0)
        sns.despine(ax=ax, offset=10, trim=True)

    # This now correctly sends 6 arguments to the 6-argument definition above
    add_custom_legend(fig, [ax1, ax2], unique_sizes, unique_times, size_to_style, time_to_color)

    plt.savefig(os.path.join(save_path, "morpho_individual_traces_clean.pdf"), dpi=200, facecolor='white',
                bbox_inches='tight')
    plt.show()


def plot_double_normalised_fitted_profiles(df, save_path):
    """Interpolated fits using self-normalization for shape comparison."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)
    plt.subplots_adjust(right=0.88, hspace=0.15)
    size_to_ax = {unique_sizes[0]: ax1, unique_sizes[1]: ax2}

    for (t, sz), group in df.groupby(["Time", "Size"]):
        ax = size_to_ax[sz]

        group = group.copy()
        group["s_norm"] = group.groupby("path")["s"].transform(lambda x: x / x.max())
        group["rho_norm"] = group.groupby("path")["rho"].transform(lambda x: x / x.max())
        bins = np.linspace(0, 1, 500)
        bin_centers = (bins[:-1] + bins[1:]) / 2
        bin_idx = np.digitize(group["s_norm"].values, bins)
        # Aggregate numeric values
        binned_mu = [group["rho_norm"].values[bin_idx == i].mean() for i in range(1, len(bins))]
        binned_mu = np.array(binned_mu)
        valid = ~np.isnan(binned_mu)

        if not np.any(valid): continue
        spline = UnivariateSpline(bin_centers[valid], binned_mu[valid], k=3, s=0.01)
        s_smooth = np.linspace(0, 1, 500)

        ax.plot(s_smooth, spline(s_smooth), color=time_to_color[t],
                linestyle=size_to_style[sz], linewidth=3.5, alpha=1.0, zorder=3)

    ax1.set_title(fr"$\mathbf{{Shape\ Comparison\ -\ {unique_sizes[0]}\ cells}}$", fontsize=14)
    ax2.set_title(fr"$\mathbf{{Shape\ Comparison\ -\ {unique_sizes[1]}\ cells}}$", fontsize=14)
    ax2.set_xlabel(r"Normalized Arc Length, $s/s_{max}$", fontsize=12)

    for ax in [ax1, ax2]:
        ax.set_ylabel(r"Relative Thickness, $\rho/\rho_{max}$", fontsize=12)
        ax.set_ylim(0, 1.1)
        sns.despine(ax=ax, offset=10, trim=True)

    add_custom_legend(fig, [ax1, ax2], unique_sizes, unique_times, size_to_style, time_to_color)
    plt.savefig(os.path.join(save_path, "morpho_shape_fits_normalized.pdf"), dpi=200, facecolor='white',
                bbox_inches='tight')
    plt.show()


# Size line styles
line_styles = ['-', '--', ':', '-.']
size_to_style = {sz: line_styles[i % len(line_styles)] for i, sz in enumerate(unique_sizes)}


# --- 2. PHYSICAL SHAPE RECONSTRUCTION ---
def plot_physical_reconstructed_silhouettes(df, save_path):
    """
    Expands normalized splines back to physical units using group averages
    and plots mirrored silhouettes to show rotational symmetry.
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 12), sharex=True)
    plt.subplots_adjust(right=0.88, hspace=0.2)
    size_to_ax = {unique_sizes[0]: ax1, unique_sizes[1]: ax2}

    for (t, sz), group in df.groupby(["Time", "Size"]):
        ax = size_to_ax[sz]

        # A. Calculate group averages for scaling
        # We find the max s and max rho for each gastruloid in the group first
        gastruloid_stats = group.groupby("path").agg(
            s_max=("s", "max"),
            rho_max=("rho", "max")
        )
        avg_s_max = gastruloid_stats["s_max"].mean()
        avg_rho_max = gastruloid_stats["rho_max"].mean()

        # B. Prepare normalized data for the spline
        group = group.copy()
        group["s_norm"] = group.groupby("path")["s"].transform(lambda x: x / x.max())
        group["rho_norm"] = group.groupby("path")["rho"].transform(lambda x: x / x.max())

        bins = np.linspace(0, 1, 300)
        bin_centers = (bins[:-1] + bins[1:]) / 2
        bin_idx = np.digitize(group["s_norm"].values, bins)

        binned_mu = [group["rho_norm"].values[bin_idx == i].mean() for i in range(1, len(bins))]
        binned_mu = np.array(binned_mu)
        valid = ~np.isnan(binned_mu)

        if not np.any(valid): continue

        # C. Generate the Spline (0-1 scale)
        spline = UnivariateSpline(bin_centers[valid], binned_mu[valid], k=3, s=0.01)
        s_norm_smooth = np.linspace(0, 1, 500)
        rho_norm_smooth = spline(s_norm_smooth)

        # D. EXPAND BACK TO PHYSICAL UNITS
        s_phys = s_norm_smooth * avg_s_max
        rho_phys = rho_norm_smooth * avg_rho_max

        # E. Plot mirrored silhouettes
        # Top half
        ax.plot(s_phys, rho_phys, color=time_to_color[t],
                linestyle=size_to_style[sz], linewidth=3, alpha=0.9, zorder=3)
        # Bottom half (mirrored)
        ax.plot(s_phys, -rho_phys, color=time_to_color[t],
                linestyle=size_to_style[sz], linewidth=3, alpha=0.9, zorder=3)

        # Optional: Fill the body for a "rendering" effect
        ax.fill_between(s_phys, -rho_phys, rho_phys, color=time_to_color[t], alpha=0.05, zorder=2)

    # Styling
    ax1.set_title(fr"$\mathbf{{Physical\ Shape\ Reconstruction\ -\ {unique_sizes[0]}\ cells}}$", fontsize=16)
    ax2.set_title(fr"$\mathbf{{Physical\ Shape\ Reconstruction\ -\ {unique_sizes[1]}\ cells}}$", fontsize=16)
    ax2.set_xlabel(r"Arc Length, $s$ ($\mu m$)", fontsize=14)

    for ax in [ax1, ax2]:
        ax.set_ylabel(r"Radius $\rho$ ($\mu m$)", fontsize=14)
        ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)  # Central axis
        sns.despine(ax=ax, offset=10, trim=True)
        # Ensure aspect ratio is equal so 1um x = 1um y (true shape)
        ax.set_aspect('equal', adjustable='datalim')

    add_custom_legend(fig, [ax1, ax2], unique_sizes, unique_times, size_to_style, time_to_color)

    plt.savefig(os.path.join(save_path, "morpho_physical_reconstruction.pdf"),
                dpi=300, facecolor='white', bbox_inches='tight')
    plt.show()


def plot_morpho_curvature_analysis(df, save_path, mode='surface'):
    """
    Plots curvature profiles using pre-calculated values.
    mode='surface' plots the Mean Curvature (average of curv_ss and curv_phiphi).
    mode='midline' plots curv_midline.
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    plt.subplots_adjust(right=0.88, hspace=0.15)
    size_to_ax = {unique_sizes[0]: ax1, unique_sizes[1]: ax2}
    title_prefix = ""
    for (t, sz), group in df.groupby(["Time", "Size"]):
        ax = size_to_ax[sz]

        # Determine which data to plot
        if mode == 'surface':
            # Mean Curvature H = (kappa_s + kappa_phi) / 2
            group['target_curv'] = (group['curv_ss'] + group['curv_phiphi']) / 2
            title_prefix = "Surface Mean Curvature"
        else:
            group['target_curv'] = group['curv_midline']
            title_prefix = "Midline Curvature"

        # Use your s_normalised key for the x-axis alignment
        bins = np.linspace(0, 1, 400)
        bin_idx = np.digitize(group["s_normalised"].values, bins)

        # Calculate mean trend and standard deviation for the 'beauty' factor
        binned_stats = group.groupby(bin_idx)['target_curv'].agg(['mean', 'std'])
        bin_centers = (bins[:-1] + bins[1:]) / 2

        # Filter for valid bins (where we have data)
        valid_indices = binned_stats.index[binned_stats.index <= len(bin_centers)]
        x_plot = bin_centers[valid_indices - 1]
        y_plot = binned_stats.loc[valid_indices, 'mean']
        y_err = binned_stats.loc[valid_indices, 'std']

        # Plot the trend line
        ax.plot(x_plot, y_plot, color=time_to_color[t],
                linestyle=size_to_style[sz], linewidth=3, alpha=0.9, zorder=3)

        # Add a subtle shadow for variability (Standard Deviation)
        ax.fill_between(x_plot, y_plot - y_err, y_plot + y_err,
                        color=time_to_color[t], alpha=0.1, zorder=2)

    # Styling
    ax1.set_title(fr"$\mathbf{{{title_prefix}\ -\ {unique_sizes[0]}\ cells}}$", fontsize=15)
    ax2.set_title(fr"$\mathbf{{{title_prefix}\ -\ {unique_sizes[1]}\ cells}}$", fontsize=15)
    ax2.set_xlabel(r"Normalized Arc Length, $s_{norm}$", fontsize=12)
    ylabel = ""
    for ax in [ax1, ax2]:
        ax.set_ylabel(ylabel, fontsize=12)
        ax.axhline(0, color='black', linewidth=0.8, alpha=0.3)
        sns.despine(ax=ax, offset=10, trim=True)

    # Unified Legend call with the [ax1, ax2] fix
    add_custom_legend(fig, [ax1, ax2], unique_sizes, unique_times, size_to_style, time_to_color)

    save_name = f"morpho_curvature_{mode}.pdf"
    plt.savefig(os.path.join(save_path, save_name), dpi=200, facecolor='white', bbox_inches='tight')
    plt.show()


# --- 1. SETUP ---
unique_times = np.sort(df_nematic_full['Time'].unique())
unique_sizes = np.sort(df_nematic_full['Size'].unique())

# Correct Modern Colormap call
cmap_ref = plt.get_cmap('coolwarm')
colors = cmap_ref(np.linspace(0, 1, len(unique_times)))
time_to_color = {t: colors[i] for i, t in enumerate(unique_times)}

line_styles = ['-', '--', ':', '-.']
size_to_style = {sz: line_styles[i % len(line_styles)] for i, sz in enumerate(unique_sizes)}


def plot_nematic_individual_profiles(df_raw, save_path, use_normalised=True):
    """
    Creates ultra-clean 4x2 grid analysis plots from raw point-wise nematic data.
    Final Optimization: 200 DPI, solid background, journal labels, and n-counts per size.
    """
    df = df_raw.copy()

    # 1. Magnitude Calculation
    df['S_magnitude'] = 2 * np.sqrt(df['Q_ss'] ** 2 + df['Q_sphi'] ** 2)

    # 2. X-axis Toggle and Journal Labels
    if use_normalised:
        df['s_plot'] = df.groupby("path")['dir_s'].transform(lambda x: x / x.max())
        x_label = r"Normalized Arc Length, $s/s_{max}$"
        suffix = "norm"
    else:
        df['s_plot'] = df['dir_s']
        x_label = r"Arc Length, $s$ ($\mu m$)"
        suffix = "phys"

    unique_times = np.sort(df['Time'].unique())
    unique_sizes = np.sort(df['Size'].unique())

    # --- Setup Figure: 4 Rows (Components) x 2 Columns (Sizes) ---
    fig, axes = plt.subplots(4, 2, figsize=(14, 14), sharex='col', sharey='row')
    plt.subplots_adjust(right=0.88, left=0.1, hspace=0.25, wspace=0.1)

    components = [
        ('S_magnitude', r'Order Magnitude, $S$', (0, 1.05)),
        ('Q_phiphi', r'Circumferential, $Q_{\phi\phi}$', (-0.55, 0.55)),
        ('Q_ss', r'Longitudinal, $Q_{ss}$', (-0.55, 0.55)),
        ('Q_sphi', r'Shear, $Q_{s\phi}$', (-0.55, 0.55))
    ]

    # --- Plotting Loop ---
    for row_idx, (col_name, label, ylims) in enumerate(components):
        for col_idx, sz_val in enumerate(unique_sizes):
            ax = axes[row_idx, col_idx]

            # Filter for specific size
            size_mask = df['Size'] == sz_val
            size_group = df[size_mask]

            # Calculate n for this specific size subgroup
            n_size = size_group['path'].nunique()

            for t_val in unique_times:
                group = size_group[size_group['Time'] == t_val]
                if group.empty: continue

                group = group.dropna(subset=['s_plot', col_name]).sort_values('s_plot')
                if len(group) < 10: continue

                # Resample and Smooth
                x_eval = np.linspace(group['s_plot'].min(), group['s_plot'].max(), 300)
                y_interp = np.interp(x_eval, group['s_plot'], group[col_name])
                y_smooth = gaussian_filter1d(y_interp, sigma=12)

                ax.plot(x_eval, y_smooth,
                        color=time_to_color[t_val],
                        linewidth=3, alpha=0.9, zorder=3)

            # --- Row Styling (Left column only) ---
            if col_idx == 0:
                ax.set_ylabel(label, fontsize=12, fontweight='bold')

            ax.set_ylim(ylims)
            if col_name != 'S_magnitude':
                ax.axhline(0, color='black', linewidth=1, alpha=0.2, linestyle='--')

            # --- Column Styling (Top and Bottom rows only) ---
            if row_idx == 0:
                ax.set_title(fr"$\mathbf{{Seeding\ Size\ {sz_val}\ (n={n_size})}}$",
                             fontsize=14, pad=20)

            if row_idx == 3:
                ax.set_xlabel(x_label, fontsize=12)

            sns.despine(ax=ax, offset=5, trim=True)

    # --- Shared Colorbar (Journal Standard) ---
    colors_list = [time_to_color[t] for t in unique_times]
    sm = cm.ScalarMappable(cmap=ListedColormap(colors_list),
                           norm=BoundaryNorm(np.arange(len(unique_times) + 1) - 0.5, len(unique_times)))

    cbar_ax = fig.add_axes([0.92, 0.35, 0.015, 0.3])
    cbar = fig.colorbar(sm, cax=cbar_ax, ticks=np.arange(len(unique_times)))
    cbar.ax.set_yticklabels([str(int(t)) for t in unique_times])
    cbar.set_label(r"Time post-seeding, $t$ (h)", fontweight='bold', fontsize=12)

    # Save Settings: 200 DPI, Solid White Background, No Transparency
    save_name = f"nematic_profile-comparison_{suffix}.pdf"
    plt.savefig(os.path.join(save_path, save_name),
                bbox_inches='tight', dpi=200, facecolor='white', transparent=False)
    plt.show()


# Ensure Matplotlib handles LaTeX-style math consistently
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.family'] = 'STIXGeneral'


def plot_nematic_phase_diagram_optimized(df_raw, save_path, component='Q_phiphi', ylims=(-0.55, 0.55), ydashed=0.0):
    """
    Optimized Publication-Ready Nematic Phase Diagram:
    - Fixed LaTeX rendering for y-labels
    - Visible underlying profiles (alpha=0.35)
    - Shaded SEM error bands
    - Journal-standard labels (t (h), lowercase n)
    - 200 DPI, solid white background
    """
    df = df_raw.copy()

    # Ensure S_magnitude exists
    if component == 'S_magnitude' and 'S_magnitude' not in df.columns:
        df['S_magnitude'] = 2 * np.sqrt(df['Q_ss'] ** 2 + df['Q_sphi'] ** 2)

    df['s_plot'] = df.groupby("path")['dir_s'].transform(lambda x: x / x.max())

    unique_times = np.sort(df['Time'].unique())
    unique_sizes = np.sort(df['Size'].unique())

    n_rows = len(unique_sizes)
    n_cols = len(unique_times)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 4 * n_rows),
                             sharex=True, sharey=True)

    if n_rows == 1: axes = np.atleast_2d(axes)
    if n_cols == 1: axes = np.atleast_2d(axes).T

    x_eval = np.linspace(0, 1, 200)

    for r, sz_val in enumerate(unique_sizes):
        for c, t_val in enumerate(unique_times):
            ax = axes[r, c]
            cell_data = df[(df['Time'] == t_val) & (df['Size'] == sz_val)].copy()

            if cell_data.empty:
                ax.axis('off')
                continue

            all_interp_profiles = []

            for file_id, gast_data in cell_data.groupby("path"):
                gast_data = gast_data.dropna(subset=['s_plot', component]).sort_values('s_plot')
                if len(gast_data) < 15: continue

                y_interp = np.interp(x_eval, gast_data['s_plot'], gast_data[component])
                y_smooth = gaussian_filter1d(y_interp, sigma=6)
                all_interp_profiles.append(y_smooth)

                ax.plot(x_eval, y_smooth, color=time_to_color[t_val], alpha=0.35, linewidth=1.0, zorder=1)

            if not all_interp_profiles:
                continue

            profile_matrix = np.array(all_interp_profiles)
            mean_line = np.nanmean(profile_matrix, axis=0)
            sem_line = np.nanstd(profile_matrix, axis=0) / np.sqrt(len(all_interp_profiles))
            n_samples = len(all_interp_profiles)

            color = time_to_color[t_val]
            ax.fill_between(x_eval, mean_line - sem_line, mean_line + sem_line,
                            color=color, alpha=0.25, zorder=2)
            ax.plot(x_eval, mean_line, color='black', linewidth=2.5, linestyle='-', zorder=3)
            ax.plot(x_eval, mean_line, color=color, linewidth=1.2, linestyle='-', zorder=4, alpha=0.7)

            ax.set_ylim(ylims)
            ax.axhline(ydashed, color='black', linewidth=1, alpha=0.2, linestyle='--')

            ax.text(0.95, 0.05, f"$n={n_samples}$", transform=ax.transAxes,
                    ha='right', va='bottom', fontsize=11, fontweight='bold',
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2))

            sns.despine(ax=ax, offset=5, trim=True)

            if r == 0:
                ax.set_title(fr"$t = {t_val}$ h", fontsize=16, pad=15)
            if c == n_cols - 1:
                ax.set_ylabel(f"Size {sz_val}", fontsize=14, fontweight='bold', labelpad=20)
                ax.yaxis.set_label_position("right")

    # --- REFINED LATEX LABELING ---
    fig.supxlabel(r"Normalized Arc Length, $s/s_{max}$", fontsize=18, y=0.02)

    # Robust string mapping for components
    if component == 'Q_phiphi':
        comp_tex = r"Q_{\phi\phi}"
    elif component == 'Q_ss':
        comp_tex = r"Q_{ss}"
    elif component == 'Q_sphi':
        comp_tex = r"Q_{s\phi}"
    elif component == 'S_magnitude':
        comp_tex = r"S"
    else:
        comp_tex = component.replace('_', r'\_')

    fig.supylabel(fr"Nematic Component: ${comp_tex}$", fontsize=18, x=0.02)

    plt.tight_layout(rect=[0.03, 0.03, 0.97, 0.97])

    save_name = f"nematic_phase_diagram_fixed_{component}.pdf"
    plt.savefig(os.path.join(save_path, save_name),
                dpi=200, bbox_inches='tight', facecolor='white', transparent=False)
    plt.show()


# --- 1. SETUP & THEME ---
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.family'] = 'STIXGeneral'
sns.set_theme(style="ticks")

# --- 2. DATA PREPARATION ---
# Ensure Body Length is present
if 'Body Length' not in df_nematic_full.columns:
    df_combined = df_nematic_full.merge(
        df_morpho_summary[['path', 'Body Length']], on='path', how='left'
    )
else:
    df_combined = df_nematic_full.copy()

# Drop rows missing critical spatial/morphological data
df_combined = df_combined.dropna(subset=['Body Length', 'dir_s'])

# Ensure scalar order parameter S is calculated if requested
if 'S' not in df_combined.columns:
    df_combined['S'] = 2 * np.sqrt(df_combined['Q_ss'] ** 2 + df_combined['Q_sphi'] ** 2)

# --- 3. BINNING BY BODY LENGTH ---
# Define physical bins (e.g., 200um to 1000um in steps of 50um)
l_min, l_max = df_combined['Body Length'].min(), df_combined['Body Length'].max()
# Rounding to clean integers for the bin edges
length_bins = np.linspace(np.floor(l_min / 50) * 50, np.ceil(l_max / 50) * 50, 20)
df_combined['Length_Bin'] = pd.cut(df_combined['Body Length'], bins=length_bins)


# --- 4. THE RE-FILLED COMPARISON ---
def plot_nematic_heatmap_comparison(df, size_left=200, size_right=300, component='S', save_path=None):
    unique_bins = df['Length_Bin'].cat.categories
    x_eval = np.linspace(0, 1, 200)  # Spatial resolution along arc length

    fig, axes = plt.subplots(1, 2, figsize=(14, 9), sharey=True, constrained_layout=True)
    im = None
    # Scale Logic
    if component == 'S':
        vmin, vmax = 0, 1
        cmap = 'magma'  # Better for 0-1 scalar fields than 'jet'
    else:
        abs_max = np.nanquantile(df[component].abs(), 0.99)
        vmin, vmax = -abs_max, abs_max
        cmap = 'RdBu_r'  # Diverging colormap for Q tensor components (positive/negative)

    target_sizes = [size_left, size_right]

    for col_idx, sz in enumerate(target_sizes):
        ax = axes[col_idx]
        size_data = df[df['Size'] == sz].copy()

        heatmap_matrix = []
        bin_labels = []

        for i, b in enumerate(unique_bins):
            # Center of the bin for labeling
            bin_labels.append(f"{int(b.mid)}")
            bin_data = size_data[size_data['Length_Bin'] == b].copy()

            if bin_data.empty:
                heatmap_matrix.append(np.full_like(x_eval, np.nan))
                continue

            # Normalize arc length within this specific bin group
            bin_data['s_norm'] = bin_data.groupby('path')['dir_s'].transform(
                lambda x: (x - x.min()) / (x.max() - x.min()))

            profiles_in_bin = []
            for _, gast_data in bin_data.groupby('path'):
                gast_data = gast_data.dropna(subset=['s_norm', component]).sort_values('s_norm')
                if len(gast_data) < 10: continue

                # Map individual gastruloid onto the common x_eval grid
                y_interp = np.interp(x_eval, gast_data['s_norm'], gast_data[component])
                profiles_in_bin.append(y_interp)

            if profiles_in_bin:
                mean_profile = np.nanmean(profiles_in_bin, axis=0)
                # Spatial smoothing for pattern clarity
                heatmap_matrix.append(gaussian_filter1d(mean_profile, sigma=2))
            else:
                heatmap_matrix.append(np.full_like(x_eval, np.nan))

        matrix = np.array(heatmap_matrix)

        # Plotting the matrix
        im = ax.imshow(matrix, aspect='auto', origin='lower',
                       extent=[0, 1, length_bins[0], length_bins[-1]],
                       cmap=cmap, vmin=vmin, vmax=vmax, interpolation='bilinear')

        # Aesthetics
        ax.set_title(fr"$\mathbf{{Seeding\ Size:\ {sz}\ Cells}}$", fontsize=18, pad=15)
        ax.set_xlabel(r"Normalized Arc Length, $s/s_{max}$", fontsize=14)

        if col_idx == 0:
            ax.set_ylabel(r"Body Length, $L$ ($\mu m$)", fontsize=14)

        # Grid lines for bin boundaries
        ax.grid(False)

    # Unified Colorbar
    cbar = fig.colorbar(im, ax=axes, location='right', shrink=0.4, pad=0.02)
    cbar.set_label(fr"$\text{{{component}}}$", fontsize=14)

    if save_path:
        save_name = f"nematic_heatmap_{component}_vs_Length.pdf"
        plt.savefig(os.path.join(save_path, save_name), dpi=300, bbox_inches='tight', facecolor='white')

    plt.show()

In [ ]:
plot_experiment_stats(df=df_morpho_full, save_path=morphology_batch_analysis_path)
plot_advanced_morphometrics(df_morpho_summary, morphology_batch_analysis_path,
                            normalise=False)
plot_individual_thickness_profiles_split(df_morpho_side, morphology_batch_analysis_path)
plot_double_normalised_fitted_profiles(df_morpho_side, morphology_batch_analysis_path)
plot_physical_reconstructed_silhouettes(df_morpho_side, morphology_batch_analysis_path)
plot_morpho_curvature_analysis(df_morpho_side, morphology_batch_analysis_path, mode='surface')
plot_morpho_curvature_analysis(df_morpho_side, morphology_batch_analysis_path, mode='midline')
plot_nematic_individual_profiles(df_nematic_full, nematic_batch_analysis_path,
                                 use_normalised=True)
plot_nematic_individual_profiles(df_nematic_full, nematic_batch_analysis_path,
                                 use_normalised=False)
components_to_plot = ['Q_phiphi', 'Q_ss', 'Q_sphi', 'S']
for comp in components_to_plot:
    if comp == 'S':
        plot_nematic_phase_diagram_optimized(df_nematic_full, nematic_batch_analysis_path,
                                             component='S_magnitude', ylims=[-0.01, 1.01], ydashed=1.0)
    else:
        plot_nematic_phase_diagram_optimized(df_nematic_full, nematic_batch_analysis_path,
                                             component=comp)
    plot_nematic_heatmap_comparison(df_combined, size_left=200, size_right=300, component=comp,
                                    save_path=nematic_batch_analysis_path)